# Diabetes Readmission — Interpretability: SHAP + Fairness

The final stop of the classification walkthrough answers two questions about the readmission
model we have built:

1. **Why does it predict what it predicts?** SHAP distributes each prediction among the
   features, grounded in cooperative game theory — the same tool used in regression form on
   `U1_RealEstate-6_Interp`, here in its classification (two-class) form.
2. **Does it work equally well for everyone?** A model that flags patients for follow-up can
   cause real clinical harm if it systematically under- or over-flags one demographic group.
   We audit the model's behavior across `race`, `gender`, and `age` with Fairlearn's
   `MetricFrame`, then try mitigations from all three families — pre-, in-, and
   post-processing.

## Learning objectives

- Compute and read SHAP values (summary, bar, waterfall, force, dependence, decision, and
  heatmap plots) for a binary classifier.
- Audit a classifier's fairness with `MetricFrame`, scalar demographic-parity/equalized-odds
  metrics, and an intersectional (race × gender) analysis.
- Compare pre-processing (`CorrelationRemover`), in-processing (`ExponentiatedGradient`,
  `GridSearch`, `AdversarialFairnessClassifier`), and post-processing (`ThresholdOptimizer`)
  mitigations on the same accuracy/fairness axes.

## Background

This notebook assumes the readmission classifier built earlier in the spine, the train/test split,
and the classification metrics from `U1_Diabetes-3_Classification`. It is the classification twin of
`U1_RealEstate-6_Interp`, which applies the same SHAP tooling to a regression model.

Two bodies of theory are used and each is developed where it is first applied: **Shapley values** in
section 2, and the **fairness criteria** (demographic parity, equalized odds) in section 3.1.

**Prerequisites:** `U1_Diabetes-3_Classification` (the model and its metrics),
`U1_Diabetes-2_Preprocess` (the dataset)

## This notebook covers

1. Load the data, filter and balance it, and train the one shared Random Forest classifier.
2. **SHAP** — TreeExplainer setup, then summary/bar/waterfall/force/dependence/decision/heatmap
   plots, all against the shared model.
3. **Fairlearn** — assessment (sensitive-feature distributions, `MetricFrame`, scalar metrics,
   intersectional analysis), then mitigation (one demo per pipeline stage), then all mitigators
   compared side by side.

**Dataset:** the **Diabetes Hospital Readmission** dataset (UCI, loaded via
`fairlearn.datasets.fetch_diabetes_hospital`) — ten years of clinical care records from 130 US
hospitals; `readmit_30_days` is the target, and `race`/`gender`/`age` are the sensitive
attributes audited in Part 3.

**References:** https://shap.readthedocs.io/ , https://fairlearn.org/


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

pd.set_option('display.max_columns',100)
pd.set_option('display.max_rows',100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# Shared course helpers (msds565_helpers.py lives in the repo root).
# Notebooks sit two folders below the root, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds565_helpers as helpers


## Imports

In [ ]:
import shap

from fairlearn.datasets import fetch_diabetes_hospital
from fairlearn.metrics import (
    MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equalized_odds_ratio,
    true_positive_rate,
    false_positive_rate,
    selection_rate,
)
# Mitigation — one tool per family
from fairlearn.preprocessing import CorrelationRemover                       # pre-processing
from fairlearn.reductions import (                                           # in-processing
    ExponentiatedGradient, GridSearch, EqualizedOdds, DemographicParity,
)
from fairlearn.adversarial import AdversarialFairnessClassifier             # in-processing (deep)
from fairlearn.postprocessing import ThresholdOptimizer                      # post-processing

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import RandomOverSampler

import tensorflow as tf   # backend for AdversarialFairnessClassifier


## 1. Dataset & shared model

We use the **Diabetes Hospital Readmission** dataset from the UCI Machine Learning Repository
(loaded via `fairlearn`). It contains ten years of clinical care records from 130 US hospitals.

| Property | Value |
|---|---|
| Samples | ~101,766 (we use a balanced, filtered sample of 20,000) |
| Features | 22 (after dropping leaky target-derived columns) |
| Target | `readmit_30_days`: 1 = readmitted within 30 days, 0 = not |
| Class balance | ~89% negative, ~11% positive — notable imbalance |

**Sensitive attributes used in the fairness audit (Part 3):**

| Attribute | Groups | Role in audit |
|---|---|---|
| `race` | AfricanAmerican, Asian, Caucasian, Hispanic, Other | Primary sensitive attribute |
| `gender` | Female, Male | Secondary sensitive attribute |
| `age` | '30 years or younger', '30–60 years', 'Over 60 years' | Secondary sensitive attribute |

> **Important:** these attributes are extracted *before* encoding and kept separate from the
> model's feature set — the model does not receive them as inputs. However, correlated clinical
> variables (diagnosis codes, medication regimens) can act as **proxies**, leading to disparate
> outcomes even without explicit use of protected attributes — which is exactly what Part 3
> audits for.

We train **one shared Random Forest classifier** below; both the SHAP explanations (Part 2) and
the fairness audit (Part 3) run against this same model.

In [ ]:
data = fetch_diabetes_hospital(as_frame=True)

df = pd.concat([data.data, data.target], axis=1)
df

In [ ]:
target = 'readmit_30_days'  # binary target (0/1)

df = df[ ~df.race.isin(['Unknown', 'Other']) ]  # remove unknown
df = df[ ~df.gender.isin(['Unknown', 'Invalid']) ]  # remove unknown

vc = df[target].value_counts()
print(f'Class distribution:\n{vc}\n')

In [ ]:
# downsample for balance and speedup
n_samp_per_class = 10000
df1 = df[df[target] == 0].sample(n_samp_per_class, random_state=42)
df2 = df[df[target] == 1].sample(n_samp_per_class, random_state=42)
df = pd.concat([df1, df2], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
print(f'Balanced class distribution:\n{df[target].value_counts()}\n')

In [ ]:
# Save the raw sensitive features BEFORE one-hot encoding
# Fairlearn MetricFrame needs the original categorical labels (e.g., 'Caucasian'),
# not the encoded binary columns, to produce human-readable group names.
sensitive_raw = df[['race', 'gender', 'age']].copy()

# Drop columns derived from the target (would cause data leakage)
X = df.drop(columns=['readmitted', 'readmit_binary', 'readmit_30_days'])
y = df[target]   # binary 0/1

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(exclude='number').columns.tolist()

# One-hot encode all categorical features (fine for tree-based models).
# NOTE: pd.get_dummies returns the numeric columns unchanged alongside the new
# dummy columns, so we assign the result directly. (Concatenating X[num_cols]
# again would silently duplicate every numeric column.)
X = pd.get_dummies(X, columns=cat_cols, drop_first=True).astype(int)

feature_names = X.columns.tolist()
n_feat = len(feature_names)

print(f'Feature matrix shape: {X.shape}  (duplicate cols: {X.columns.duplicated().any()})')
print(f'Sensitive features saved: {sensitive_raw.columns.tolist()}')
X.head()

### 1.1 Train the shared baseline model

We train a **Random Forest** classifier on the encoded feature set — this is the one baseline
model both Part 2 (SHAP) and Part 3 (Fairlearn) audit below. It does *not* receive `race`,
`gender`, or `age` as explicit inputs (those columns are held out as `sensitive_raw`, not
concatenated into `X`), but it may still learn correlated clinical proxies that produce
disparate outcomes — **proxy discrimination**, one of the most common real-world fairness
failures, and exactly what Part 3 checks for.

We oversample the minority class in the training fold only (to handle class imbalance), then
evaluate on the original held-out test set. `TreeExplainer` (Part 2) is optimized for tree
ensembles like this one — it computes exact Shapley values by exploiting the tree structure in
polynomial time, far faster than model-agnostic Kernel SHAP.

In [ ]:
# Pass sensitive_raw through the split so it stays perfectly aligned with X and y.
# Using train_test_split with multiple arrays guarantees identical row ordering.
X_train, X_test, y_train, y_test, sf_train, sf_test = train_test_split(
    X, y, sensitive_raw,
    test_size=0.2, stratify=y, random_state=42
)

# Reset indices to 0-based so Fairlearn can align arrays reliably
X_train  = X_train.reset_index(drop=True)
X_test   = X_test.reset_index(drop=True)
y_train  = y_train.reset_index(drop=True)
y_test   = y_test.reset_index(drop=True)
sf_train = sf_train.reset_index(drop=True)
sf_test  = sf_test.reset_index(drop=True)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')
print(f'sf_train: {sf_train.shape}  |  sf_test: {sf_test.shape}')

In [ ]:
# Oversample the minority class in the training fold only
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

# Train the baseline Random Forest
model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
model.fit(X_train_res, y_train_res)

# Baseline predictions on the held-out test set
y_pred = model.predict(X_test)

print('=== BASELINE MODEL ===')
print(f'Training shape: {y_train.value_counts().to_dict()}')
print(f'Resampled shape: {y_train_res.value_counts().to_dict()}\n')
print(classification_report(y_test, y_pred, target_names=['Not Readmitted', 'Readmitted']))

## 2. Why does it predict what it predicts? — SHAP

**SHAP.** For a single prediction $f(x)$, Shapley values fairly distribute credit among all
features:

$$f(x) = \phi_0 + \sum_{j=1}^{p} \phi_j$$

where $\phi_0$ is the **base value** (mean model output over the training set) and $\phi_j$ is
the **SHAP value** for feature $j$ — how much that feature added to or subtracted from $\phi_0$
to produce the actual prediction. Shapley values satisfy four axioms that make them uniquely
fair:

| Axiom | Meaning |
|---|---|
| **Efficiency** | SHAP values sum exactly to $f(x) - \phi_0$ |
| **Symmetry** | Features with identical contributions get equal credit |
| **Dummy** | A feature that never changes the prediction gets SHAP = 0 |
| **Additivity** | SHAP values for an ensemble equal the sum of values for each tree |

**Fairlearn.** A model can look excellent in aggregate while quietly treating subpopulations
very differently — and even when a model never sees `race`, `gender`, or `age` directly, it can
reconstruct them from correlated variables (diagnosis codes, medication regimens), so *dropping*
a protected attribute is not sufficient for fairness. Fairlearn organizes its functionality into
two pillars:

| Pillar | Question it answers | Key tools |
|---|---|---|
| **Assessment** | *How unfair is the model, and to whom?* | `MetricFrame`, `demographic_parity_*`, `equalized_odds_*` |
| **Mitigation** | *How do we reduce the disparity?* | `CorrelationRemover` (pre), `ExponentiatedGradient` / `GridSearch` / `AdversarialFairnessClassifier` (in), `ThresholdOptimizer` (post) |

The three major fairness criteria:

| Criterion | Intuition | Formula |
|---|---|---|
| **Demographic Parity** | Every group gets positive predictions at the same rate | $P(\hat{Y}=1 \mid A=a) = P(\hat{Y}=1 \mid A=b)$ |
| **Equal Opportunity** | True positive rate (recall) is equal across groups | $P(\hat{Y}=1 \mid Y=1, A=a) = P(\hat{Y}=1 \mid Y=1, A=b)$ |
| **Equalized Odds** | Both TPR *and* FPR are equal across groups | TPR parity **and** FPR parity simultaneously |

> **Impossibility theorem (Chouldechova 2017):** when base rates differ across groups, you
> *cannot* simultaneously satisfy demographic parity, equalized odds, and calibration — choosing
> a fairness criterion is a *value judgment*, not a technical one.

In short: SHAP answers "why *this* prediction" by splitting the gap between the model's average
output and this particular output across the features, in the one way that satisfies those axioms.
Everything in this section is a different view of the same $\phi_j$ values — per-prediction
(waterfall, force), aggregated (bar, beeswarm), or against a feature's value (dependence).

### 2.1 TreeExplainer Setup

`shap.TreeExplainer` computes exact Shapley values for tree ensembles. For binary classification in SHAP ≥ 0.44, both the old and new APIs return values for **both classes** in a single array — the last axis indexes the class.

We focus on **class 1** (readmitted within 30 days) throughout.

| Interface | Returns | Shape | Used for |
|---|---|---|---|
| `explainer.shap_values(X)` | NumPy array | `(n, p, 2)` | `summary_plot`, `dependence_plot`, `decision_plot`, `force_plot` |
| `explainer(X)` | `Explanation` object | `(n, p, 2)` | `plots.bar`, `plots.beeswarm`, `plots.waterfall`, `plots.heatmap` |

> **Note:** Feeding a **DataFrame** (rather than a NumPy array) to `explainer(X)` automatically propagates column names into the `Explanation` object — required by `plots.heatmap` and recommended for all new-API plots.

In [ ]:
explainer = shap.TreeExplainer(model)

# Old API — returns ndarray (n_test, n_features, 2) in SHAP >= 0.44
# Index last axis for class: [:, :, 0] = class 0, [:, :, 1] = class 1
sv_raw   = explainer.shap_values(X_test)   # (n_test, n_features, 2)
sv       = sv_raw[:, :, 1]                  # class 1 → (n_test, n_features)
base_val = explainer.expected_value[1]

# New API — feed DataFrame so feature names propagate into the Explanation object
sv_exp = explainer(X_test)                  # Explanation (n_test, n_features, 2)
sv_new = sv_exp[:, :, 1]                    # Explanation for class 1

print(f'SHAP matrix shape (old API):  {sv.shape}')
print(f'Base value E[f(x)]:           {base_val:.4f}')
print(f'\nSanity check (first test sample):')
print(f'  base_val + sum(SHAP values) = {base_val + sv[0].sum():.4f}')
print(f'  model predict_proba class 1 = {model.predict_proba(X_test.iloc[:1])[0, 1]:.4f}')

### 2.2 Summary Plots — Dot/Beeswarm and Violin (Classic API)

The classic `shap.summary_plot` function offers two variants:

- **Dot** — identical to the beeswarm; each dot is one sample
- **Violin** — kernel density estimation of the SHAP distribution per feature; cleaner when sample count is large and dots heavily overlap

Both plots use the same red–blue color scale mapping raw feature values.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(25, 25))

plt.sca(axes[0])
shap.summary_plot(sv, X_test, feature_names=feature_names,
                  max_display=n_feat, plot_type='dot', show=False)
axes[0].set_title('Dot summary')

plt.sca(axes[1])
shap.summary_plot(sv, X_test, feature_names=feature_names,
                  max_display=n_feat, plot_type='violin', show=False)
axes[1].set_title('Violin summary')

plt.tight_layout()
plt.show()

### 2.3 Bar Plot — Global Feature Importance

The bar plot shows **mean |SHAP|** for each feature — the average magnitude of that feature's contribution across all test samples. This is the SHAP analogue of Gini importance, but unbiased toward high-cardinality features and computed from the actual Shapley attribution rather than split statistics.

Features with mean |SHAP| near zero contributed almost nothing on average — they are good candidates for removal without harming predictive accuracy.

In [ ]:
shap.plots.bar(sv_new, max_display=n_feat)

### 2.4 Waterfall Plots — Four Representative Cases

The waterfall plot explains a *single* prediction step by step. Starting from the base value $E[f(x)]$, each bar extends left (negative — pushes toward *not* readmitted) or right (positive — pushes toward readmission within 30 days). The final value is the model's output for that patient.

We display waterfall plots for all four classification outcomes — True Positive, True Negative, False Positive, and False Negative — so we can compare the feature contributions that drive correct predictions against those behind each error type.

In [ ]:
# Helper to grab a random index from a boolean mask
def get_random_idx(mask):
    indices = np.where(mask)[0]
    return np.random.choice(indices) if len(indices) > 0 else None

# Define masks for the four cases
tp_mask = (y_test.values == 1) & (y_pred == 1)
tn_mask = (y_test.values == 0) & (y_pred == 0)
fp_mask = (y_test.values == 0) & (y_pred == 1)
fn_mask = (y_test.values == 1) & (y_pred == 0)

# Select indices
indices = {
    "True Positive": get_random_idx(tp_mask),
    "True Negative": get_random_idx(tn_mask),
    "False Positive": get_random_idx(fp_mask),
    "False Negative": get_random_idx(fn_mask)
}

# 1. Iterate through your selected indices
for label, idx in indices.items():
    if idx is not None:
        # Create a new, fresh figure for every case
        plt.figure(figsize=(15, 10)) 
        
        # Explain the prediction
        # If multiclass, ensure you slice: sv_new[idx, :, 1]
        shap.plots.waterfall(sv_new[idx], max_display=12, show=False)
        
        # Add metadata title
        plt.title(
            f'{label} (Patient Index: {idx})\n'
            f'Ground Truth: {y_test.values[idx]} | Model Prediction: {y_pred[idx]}', 
            fontsize=16, 
            pad=20
        )
        
        plt.tight_layout()
        plt.show() # This renders the individual figure before moving to the next iteration
    else:
        print(f"Skipping {label}: No patients found in this category.")

### 2.5 Force Plots — Four Representative Cases

The force plot is a compact alternative to the waterfall. We generate one for each of the four classification outcomes (TP / TN / FP / FN), making it easy to compare how the model decomposes predictions across all quadrants of the confusion matrix. Features in **red** push the output above the base value (toward readmission); features in **blue** push it below (away from readmission). The predicted value sits at the boundary between the two colored groups.

Force plots are well suited for embedding in clinical reports or dashboards where space is limited.

In [ ]:
# Assuming 'indices' and 'sv' (SHAP values) are already defined from the previous step
for label, idx in indices.items():
    if idx is not None:
        # shap.force_plot with matplotlib=True handles its own figure creation.
        # We pass show=False so we can add the title before rendering.
        shap.force_plot(
            base_val, 
            sv[idx], 
            X_test.iloc[idx], 
            feature_names=feature_names,
            matplotlib=True, 
            show=False,
            figsize=(10, 5)
        )
        
        # Access the current figure to add the specific title
        plt.title(
            f'{label} — Patient {idx}\n'
            f'True Label: {y_test.values[idx]} | Model Prediction: {y_pred[idx]}',
            fontsize=14,
            pad=20
        )
        
        # Adjust layout and show the individual figure
        plt.tight_layout()
        plt.show()
    else:
        print(f"Skipping {label}: No patients found.")

### 2.6 Dependence Plots

A **dependence plot** for feature $j$ shows:

- **X-axis** — the raw value of feature $j$
- **Y-axis** — the SHAP value for feature $j$ (its marginal contribution to the prediction)
- **Color** — a second feature auto-selected by SHAP to reveal interactions (the feature most correlated with the residual variation in SHAP$_j$)

A clean monotone curve means $j$ has a simple, roughly linear effect. Two color-banded streams indicate an **interaction**: the effect of $j$ on the prediction depends on the value of the colored feature.

We plot the six most important features by mean |SHAP|.

In [ ]:
top_feats = np.argsort(-np.abs(sv).mean(axis=0))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for plot_i, feat_idx in enumerate(top_feats):
    ax = axes[plot_i // 3, plot_i % 3]
    shap.dependence_plot(
        int(feat_idx), sv, X_test,
        feature_names=feature_names,
        ax=ax, show=False
    )
    ax.set_title(f'Dependence: {feature_names[feat_idx]}')

plt.tight_layout()
plt.show()

### 2.7 Decision Plot

The **decision plot** shows the full prediction path for multiple patients simultaneously. Each line is one patient; the x-axis accumulates SHAP contributions feature by feature (sorted top-to-bottom by mean |SHAP|). Lines diverge where patients disagree on a feature's contribution.

Misclassified patients are highlighted in orange — their trajectories often diverge from the correctly-classified majority at a specific feature, pinpointing where the model went wrong.

In [ ]:
n_show = 60
misclassified = np.where(y_pred[:n_show] != y_test.values[:n_show])[0]

shap.decision_plot(
    base_val,
    sv[:n_show],
    feature_names=feature_names,
    highlight=list(misclassified) if len(misclassified) else None,
    legend_labels=['Misclassified'] if len(misclassified) else None,
    title=f'Decision plot — first {n_show} test patients'
)
plt.show()

### 2.8 Heatmap Plot

The **heatmap** places patients on the x-axis and features on the y-axis. Each cell is colored by the SHAP value for that (patient, feature) pair. Patients are sorted left-to-right by predicted readmission probability, so the pattern transitions smoothly from low-risk (left) to high-risk (right).

The thin bar at the bottom shows the model output for each patient. Horizontal bands that are consistently red or blue identify features with a strong, consistent directional effect across the entire cohort.

> **Implementation note:** `shap.plots.heatmap` requires the `Explanation` object to carry `feature_names`. This is guaranteed when a **DataFrame** (not a NumPy array) is passed to `explainer()` — the column names propagate automatically.

In [ ]:
prob_order = np.argsort(model.predict_proba(X_test)[:, 1])
sv_sorted  = sv_new[prob_order]

shap.plots.heatmap(sv_sorted, max_display=n_feat)

## 3. Does it work equally well for everyone? — Fairlearn

*How unfair is the model, and to whom?* We start by **measuring** — the model is not changed in
sections 3.1–3.5 below. Let $\hat{Y}$ be the model's prediction, $Y$ the ground truth, and $A$ a
sensitive attribute.

### 3.1 Fairness Criteria in Depth

#### 3.1.1 Demographic Parity

All groups should receive the positive prediction at the same rate:
$$\text{DP difference} = \max_a P(\hat{Y}=1 \mid A=a) - \min_a P(\hat{Y}=1 \mid A=a)$$

This criterion is *output-focused* — it cares about the distribution of predictions, not whether they are correct.

#### 3.1.2 Equal Opportunity

Among patients who *were* readmitted ($Y=1$), every group should have equal recall:
$$\text{TPR difference} = \max_a P(\hat{Y}=1 \mid Y=1, A=a) - \min_a P(\hat{Y}=1 \mid Y=1, A=a)$$

This is the right criterion when **false negatives are the costly error** (e.g., missing a high-risk patient).

#### 3.1.3 Equalized Odds

Both the true positive rate *and* the false positive rate must be equal across groups:
$$\text{EO difference} = \max\left(\text{TPR difference},\ \text{FPR difference}\right)$$

This is strictly stronger than equal opportunity. It ensures no group is disproportionately flagged (high FPR) *or* missed (low TPR).

#### 3.1.4 Difference vs. Ratio

Fairlearn reports fairness gaps as both a **difference** and a **ratio**:

| Metric form | Formula | Ideal value | Common threshold |
|---|---|---|---|
| Difference | $\max - \min$ across groups | 0 | $\leq 0.10$ |
| Ratio | $\min / \max$ across groups | 1 | $\geq 0.80$ (the '80% rule') |

### 3.2 Sensitive Feature Distributions

A fairness audit begins with population accounting: how many individuals from each group appear in the test set? Small groups yield high-variance metric estimates — keep group sizes in mind when interpreting results.

In [ ]:
# Reusable color palette for all plots in this notebook
colors = plt.cm.tab10.colors

### 3.3 MetricFrame: Disaggregated Metrics by Group

`MetricFrame` is Fairlearn's primary audit tool. It takes any set of sklearn-compatible metric functions and evaluates each one **separately for every group** defined by a sensitive feature.

#### 3.3.1 Key properties

| Property / Method | Description |
|---|---|
| `.overall` | Metrics computed on the full dataset (ignoring groups) |
| `.by_group` | Metrics broken out per group as a DataFrame |
| `.difference()` | Max − min across groups per metric (0 = perfect parity) |
| `.ratio()` | Min / max across groups per metric (1 = perfect parity) |

#### 3.3.2 Metrics we audit

| Metric | Fairness concern it addresses |
|---|---|
| **Accuracy** | Are some groups harder to classify correctly? |
| **Selection rate** | Does the model favor certain groups with positive predictions? |
| **True positive rate** | Does the model miss readmissions more in some groups? |
| **False positive rate** | Does the model over-flag certain groups as at-risk? |

We run separate MetricFrame instances for `race`, `gender`, and `age`.

In [ ]:
#'race', 'gender', 'age'
col = 'race'
counts = sf_test[col].value_counts()
pcts   = (counts / len(sf_test) * 100).round(1)
summary = pd.DataFrame({'count': counts, 'pct (%)': pcts})
print(f'--- {col.upper()} ---')
print(summary.to_string())
print()

# Define the set of metrics once and reuse across all MetricFrame calls
metrics_dict = {
    'accuracy':            accuracy_score,
    'selection_rate':      selection_rate,
    'true_positive_rate':  true_positive_rate,
    'false_positive_rate': false_positive_rate,
}

# --- Audit by RACE ---
mf_race = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test['race']
)

print('=== METRICS BY RACE ===')
print('\nOverall (full test set):')
print(mf_race.overall.round(3).to_string())
print('\nDisaggregated by race:')
print(mf_race.by_group.round(3).sort_values('accuracy', ascending=False).to_string())
print('\nDifference (max − min):')
print(mf_race.difference().round(3).to_string())
print('\nRatio (min / max):')
print(mf_race.ratio().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Disaggregated Metrics by Race — Baseline Model', fontsize=14, y=1.01)

for ax, metric in zip(axes.flat, metrics_dict.keys()):
    vals = mf_race.by_group[metric].sort_values()
    bars = ax.bar(vals.index, vals.values, color=colors[:len(vals)])
    ax.axhline(mf_race.overall[metric], color='white', linestyle='--', linewidth=1.5,
               label=f'Overall: {mf_race.overall[metric]:.3f}')
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Race')
    ax.set_ylabel('Value')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
#'race', 'gender', 'age'
col = 'gender'
counts = sf_test[col].value_counts()
pcts   = (counts / len(sf_test) * 100).round(1)
summary = pd.DataFrame({'count': counts, 'pct (%)': pcts})
print(f'--- {col.upper()} ---')
print(summary.to_string())
print()

# --- Audit by GENDER ---
mf_gender = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test['gender']
)

print('=== METRICS BY GENDER ===')
print('\nOverall:')
print(mf_gender.overall.round(3).to_string())
print('\nDisaggregated by gender:')
print(mf_gender.by_group.round(3).sort_values('accuracy', ascending=False).to_string())
print('\nDifference (max − min):')
print(mf_gender.difference().round(3).to_string())
print('\nRatio (min / max):')
print(mf_gender.ratio().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
fig.suptitle('Disaggregated Metrics by Gender — Baseline Model', fontsize=14, y=1.01)

for ax, metric in zip(axes.flat, metrics_dict.keys()):
    vals = mf_gender.by_group[metric].sort_values()
    bars = ax.bar(vals.index, vals.values, color=colors[:len(vals)])
    ax.axhline(mf_gender.overall[metric], color='white', linestyle='--', linewidth=1.5,
               label=f'Overall: {mf_gender.overall[metric]:.3f}')
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Gender')
    ax.set_ylabel('Value')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
#'race', 'gender', 'age'
col = 'age'
counts = sf_test[col].value_counts()
pcts   = (counts / len(sf_test) * 100).round(1)
summary = pd.DataFrame({'count': counts, 'pct (%)': pcts})
print(f'--- {col.upper()} ---')
print(summary.to_string())
print()

# --- Audit by AGE ---
mf_age = MetricFrame(
    metrics=metrics_dict,
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test['age']
)

print('=== METRICS BY AGE GROUP ===')
print('\nOverall:')
print(mf_age.overall.round(3).to_string())
print('\nDisaggregated by age:')
print(mf_age.by_group.round(3).to_string())
print('\nDifference (max − min):')
print(mf_age.difference().round(3).to_string())
print('\nRatio (min / max):')
print(mf_age.ratio().round(3).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Disaggregated Metrics by Age Group — Baseline Model', fontsize=14, y=1.01)

for ax, metric in zip(axes.flat, metrics_dict.keys()):
    vals = mf_age.by_group[metric].sort_values()
    bars = ax.bar(vals.index, vals.values, color=colors[:len(vals)])
    ax.axhline(mf_age.overall[metric], color='white', linestyle='--', linewidth=1.5,
               label=f'Overall: {mf_age.overall[metric]:.3f}')
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlabel('Age Group')
    ax.set_ylabel('Value')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

### 3.4 Scalar Fairness Metrics

MetricFrame tables show the full picture, but practitioners often need a **single number** to compare models or set pass/fail thresholds. Fairlearn provides scalar summary functions for the most important fairness criteria.

| Function | Measures | Ideal |
|---|---|---|
| `demographic_parity_difference` | Gap in positive prediction rates | 0 |
| `demographic_parity_ratio` | Ratio of lowest to highest selection rate | 1 |
| `equalized_odds_difference` | Max of (TPR gap, FPR gap) across groups | 0 |
| `equalized_odds_ratio` | Min of (TPR ratio, FPR ratio) across groups | 1 |

We compute these for all three sensitive attributes and assemble a summary table for easy comparison.

#### 3.4.1 Interpretation guide
- **Difference ≤ 0.10** → generally considered acceptable in practice
- **Ratio ≥ 0.80** → the ‘80% rule’ from US employment law (Uniform Guidelines on Employee Selection Procedures)

In [ ]:
sensitive_cols = ['race', 'gender', 'age']
rows = []

for col in sensitive_cols:
    sf = sf_test[col]
    rows.append({
        'sensitive_feature': col,
        'dp_difference':  round(demographic_parity_difference(y_test, y_pred, sensitive_features=sf), 4),
        'dp_ratio':       round(demographic_parity_ratio(y_test, y_pred, sensitive_features=sf), 4),
        'eo_difference':  round(equalized_odds_difference(y_test, y_pred, sensitive_features=sf), 4),
        'eo_ratio':       round(equalized_odds_ratio(y_test, y_pred, sensitive_features=sf), 4),
    })

scalar_df = pd.DataFrame(rows).set_index('sensitive_feature')
print('=== SCALAR FAIRNESS METRICS — BASELINE MODEL ===')
print(scalar_df.to_string())
print()
print('Reference:  difference ≤ 0.10 = acceptable  |  ratio ≥ 0.80 = 80% rule')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Scalar Fairness Metrics by Sensitive Feature — Baseline Model', fontsize=13)

# Difference metrics (lower = fairer)
scalar_df[['dp_difference', 'eo_difference']].plot(
    kind='bar', ax=axes[0], color=['steelblue', 'coral'], width=0.6
)
axes[0].axhline(0.10, color='yellow', linestyle='--', linewidth=1.5, label='Threshold (0.10)')
axes[0].set_title('Difference Metrics (lower = fairer)')
axes[0].set_ylabel('Difference')
axes[0].set_ylim(0, 0.6)
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend()

# Ratio metrics (higher = fairer)
scalar_df[['dp_ratio', 'eo_ratio']].plot(
    kind='bar', ax=axes[1], color=['steelblue', 'coral'], width=0.6
)
axes[1].axhline(0.80, color='yellow', linestyle='--', linewidth=1.5, label='80% rule (0.80)')
axes[1].set_title('Ratio Metrics (higher = fairer)')
axes[1].set_ylabel('Ratio')
axes[1].set_ylim(0, 1.15)
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.tight_layout()
plt.show()

### 3.5 Intersectional Fairness Analysis

Single-attribute audits can **mask discrimination** that only appears at the intersection of multiple protected characteristics — a phenomenon studied by legal scholar Kimberlé Crenshaw under the term *intersectionality*.

**Why intersectional analysis matters:**
- A race audit might show small disparities between Black and white patients overall
- A gender audit might also look acceptable
- But *Black women* specifically could face a far larger disadvantage, hidden because white women and Black men bring the group averages closer together

We create a combined `race × gender` label and audit at that granularity. This surface has more groups and smaller counts per cell — interpret with appropriate caution for small groups.

In [ ]:
sf_test_inter = sf_test.copy()
sf_test_inter['race_x_gender'] = (
    sf_test_inter['race'].str.strip("'") + ' / ' + sf_test_inter['gender']
)

mf_inter = MetricFrame(
    metrics={
        'accuracy':            accuracy_score,
        'selection_rate':      selection_rate,
        'true_positive_rate':  true_positive_rate,
        'false_positive_rate': false_positive_rate,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=sf_test_inter['race_x_gender']
)

print('=== INTERSECTIONAL ANALYSIS: Race × Gender ===')
print('\nDisaggregated metrics:')
print(mf_inter.by_group.round(3).sort_values('accuracy', ascending=False).to_string())
print('\nDifferences (max − min):')
print(mf_inter.difference().round(3).to_string())

In [ ]:
by_group_inter = mf_inter.by_group.sort_values('selection_rate', ascending=False)
n_groups = len(by_group_inter)

inter_metrics  = ['selection_rate', 'true_positive_rate', 'false_positive_rate']
inter_colors   = ['steelblue', 'seagreen', 'coral']
inter_titles   = ['Selection Rate', 'True Positive Rate', 'False Positive Rate']

fig, axes = plt.subplots(3, 1, figsize=(14, 13))
fig.suptitle('Intersectional Fairness: Race × Gender — Baseline Model', fontsize=14, y=1.01)

for ax, metric, color, title in zip(axes, inter_metrics, inter_colors, inter_titles):
    vals = by_group_inter[metric]
    bars = ax.bar(range(n_groups), vals.values, color=color, alpha=0.85)
    ax.axhline(mf_inter.overall[metric], color='white', linestyle='--', linewidth=1.2,
               label=f'Overall: {mf_inter.overall[metric]:.3f}')
    ax.set_xticks(range(n_groups))
    ax.set_xticklabels(vals.index, rotation=35, ha='right', fontsize=8)
    ax.set_title(title)
    ax.set_ylabel('Value')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.show()

### 3.6 A Binary Sensitive Attribute for Mitigation

*How do we reduce the disparity?* Each mitigation algorithm intervenes at a different stage of the ML pipeline. We demonstrate **one tool per family** and re-audit after each.

| Family | Tool | Acts on |
|---|---|---|
| **Pre-processing** | `CorrelationRemover` | the **data**, before training |
| **In-processing** | `ExponentiatedGradient`, `GridSearch`, `AdversarialFairnessClassifier` | the **training objective** |
| **Post-processing** | `ThresholdOptimizer` | the **trained model's decisions** |

Section 3.2–3.5 audited the full, multi-valued `race`, `gender`, and `age`. For **mitigation** we focus on **age**, collapsed to a **binary** attribute — `Over 60` vs. `60 or younger`:

- Age showed the clearest, most *stably estimated* disparity above — the model over-flags older patients (a higher false-positive rate). Race's largest gaps sat in very small groups (Asian, Unknown) that are noisy and wash out once collapsed to a binary.
- Reduction algorithms (especially `GridSearch`) also scale poorly with the number of groups, so a binary attribute keeps them tractable and matches the "Group A vs. Group B" framing common in the literature.

Auditing stays granular; mitigation stays tractable. All mitigators and comparisons below use this binary attribute on the same standardized features.

In [ ]:
# Binary sensitive attribute used throughout the rest of this section: age (Over 60 vs not).
# The audit above showed the clearest, most stably-estimated disparity is in AGE - the model
# over-flags older patients (higher FPR). Race's largest gaps lived in tiny groups
# (Asian, Unknown) that are noisy and wash out when collapsed to a binary, so age gives
# the cleaner mitigation demonstration.
sfb_train = np.where(sf_train['age'] == "'Over 60 years'", 'Over 60', '60 or younger')
sfb_test  = np.where(sf_test['age']  == "'Over 60 years'", 'Over 60', '60 or younger')

print('Binary group sizes (test):')
print(pd.Series(sfb_test).value_counts().to_string())

# Several mitigators wrap a LINEAR base learner (LogisticRegression) and refit it many
# times, so we standardize features once up front. Tree models don't need scaling, but
# the reductions and the adversarial network converge far better on standardized inputs.
scaler = StandardScaler()
X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X_test.columns,  index=X_test.index)

# Baseline gaps on the BINARY attribute, for reference in every section below
eo_base = equalized_odds_difference(y_test, y_pred, sensitive_features=sfb_test)
dp_base = demographic_parity_difference(y_test, y_pred, sensitive_features=sfb_test)
print()
print(f'Baseline (binary age)  ->  accuracy {accuracy_score(y_test, y_pred):.3f} | '
      f'EO diff {eo_base:.3f} | DP diff {dp_base:.3f}')

### 3.7 Mitigation — Pre-processing: `CorrelationRemover`

**Family:** pre-processing (acts on the data, before any model is trained).

`CorrelationRemover` applies a linear transformation to the non-sensitive feature columns that **strips out their linear correlation** with the sensitive columns. The intent is *fairness through blindness done right*: once the features no longer carry a linear signal about age, we can drop the age columns and train a model that cannot easily use them as **proxies**.

**How it works**
1. Identify the sensitive columns (here, the one-hot `age_*` columns).
2. Regress every other feature on the sensitive columns and keep only the **residual** — the part uncorrelated with age.
3. Output the residual features (sensitive columns removed); train as usual.

**Limitations** — it only removes *linear* correlation, so non-linear proxies can survive; and decorrelating features can cost accuracy. We retrain the same Random Forest on the decorrelated features and re-audit.

In [ ]:
age_cols = [c for c in X.columns if c.startswith('age_')]

# 1. Fit the remover on the TRAIN fold and transform both folds.
#    Output drops the sensitive (age_*) columns and decorrelates the rest from them.
cr = CorrelationRemover(sensitive_feature_ids=age_cols)
cr.fit(X_train)
cr_cols = [c for c in X_train.columns if c not in age_cols]
X_train_cr = pd.DataFrame(cr.transform(X_train), columns=cr_cols, index=X_train.index)
X_test_cr  = pd.DataFrame(cr.transform(X_test),  columns=cr_cols, index=X_test.index)
print(f'Removed {len(age_cols)} age columns; decorrelated {len(cr_cols)} features.')

# 2. Same training recipe as the baseline (oversample minority, then RandomForest).
X_train_cr_res, y_train_cr_res = RandomOverSampler(random_state=42).fit_resample(X_train_cr, y_train)
model_cr = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
model_cr.fit(X_train_cr_res, y_train_cr_res)
y_pred_cr = model_cr.predict(X_test_cr)

# 3. Re-audit on the binary attribute.
print()
print('=== CorrelationRemover (pre-processing) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_cr):.3f}  (baseline {accuracy_score(y_test, y_pred):.3f})')
print(f'EO diff  : {equalized_odds_difference(y_test, y_pred_cr, sensitive_features=sfb_test):.3f}  (baseline {eo_base:.3f})')
print(f'DP diff  : {demographic_parity_difference(y_test, y_pred_cr, sensitive_features=sfb_test):.3f}  (baseline {dp_base:.3f})')

### 3.8 Mitigation — In-processing (Reductions & Adversarial)

These methods need the sensitive attribute **during training**, but — unlike post-processing — not at prediction time. We wrap a fast linear base learner (`LogisticRegression`) for the two **reduction** methods, then finish with a neural **adversarial** approach.

#### 3.8.1 `ExponentiatedGradient`

Reductions reframe fair classification as a **constrained optimization game**: a predictor tries to maximize accuracy while a "regulator" penalizes fairness-constraint violations. `ExponentiatedGradient` iteratively re-weights the training points where the constraint is violated and refits the base learner, converging to a randomized ensemble that satisfies the constraint.

We use the **`EqualizedOdds`** constraint (swap in `DemographicParity()` for selection-rate parity instead).

In [ ]:
eg = ExponentiatedGradient(
    estimator=LogisticRegression(max_iter=1000),
    constraints=EqualizedOdds(),
    max_iter=50,
)
eg.fit(X_train_sc, y_train, sensitive_features=sfb_train)
y_pred_eg = eg.predict(X_test_sc)

print('=== ExponentiatedGradient (in-processing, EqualizedOdds) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_eg):.3f}  (baseline {accuracy_score(y_test, y_pred):.3f})')
print(f'EO diff  : {equalized_odds_difference(y_test, y_pred_eg, sensitive_features=sfb_test):.3f}  (baseline {eo_base:.3f})')
print(f'DP diff  : {demographic_parity_difference(y_test, y_pred_eg, sensitive_features=sfb_test):.3f}  (baseline {dp_base:.3f})')

#### 3.8.2 `GridSearch` and the Pareto Frontier

Where `ExponentiatedGradient` returns a *single* constrained model, `GridSearch` trains a **sequence** of models across a grid of fairness-penalty weights (Lagrange multipliers). Each model is a different **accuracy ↔ fairness** compromise. Plotting them traces the **Pareto frontier** — the set of best-achievable tradeoffs — letting a practitioner pick the point that fits their risk tolerance.

We score every model on the test set and select the one with the best accuracy-minus-EO balance.

In [ ]:
gs = GridSearch(
    estimator=LogisticRegression(max_iter=1000),
    constraints=EqualizedOdds(),
    grid_size=50,
)
gs.fit(X_train_sc, y_train, sensitive_features=sfb_train)

# Evaluate every predictor on the frontier
gs_acc = np.array([accuracy_score(y_test, p.predict(X_test_sc)) for p in gs.predictors_])
gs_eo  = np.array([equalized_odds_difference(y_test, p.predict(X_test_sc),
                                             sensitive_features=sfb_test) for p in gs.predictors_])

# Select the knee: best (accuracy − EO difference)
best = int(np.argmax(gs_acc - gs_eo))
y_pred_gs = gs.predictors_[best].predict(X_test_sc)

print('=== GridSearch (in-processing) — frontier of', len(gs.predictors_), 'models ===')
print(f'Selected model →  accuracy {gs_acc[best]:.3f} | EO diff {gs_eo[best]:.3f}')

# Plot the Pareto frontier
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(gs_eo, gs_acc, s=70, color='steelblue', alpha=0.8, label='GridSearch models')
ax.scatter(gs_eo[best], gs_acc[best], s=240, marker='*', color='gold',
           edgecolor='black', zorder=5, label='Selected (knee)')
ax.scatter(eo_base, accuracy_score(y_test, y_pred), s=160, marker='X', color='coral',
           edgecolor='black', zorder=5, label='Baseline RF')
ax.set_xlabel('Equalized-odds difference  (← fairer)')
ax.set_ylabel('Accuracy  (higher = better ↑)')
ax.set_title('GridSearch — Accuracy vs. Fairness Pareto Frontier')
ax.legend()
plt.tight_layout()
plt.show()

#### 3.8.3 `AdversarialFairnessClassifier`

The most powerful — and most finicky — in-processing method. Two neural networks train against each other:

- a **predictor** that learns to predict readmission, and
- an **adversary** that tries to recover the sensitive attribute from the predictor's output.

The predictor is rewarded for *fooling* the adversary, so it is pushed toward a representation that is predictive of the label yet **uninformative about race**. The `alpha` hyperparameter trades off the two objectives.

> **Caveat — read before interpreting the output.** Adversarial training is unstable: with a too-strong adversary or too-high a learning rate, the predictor *collapses* to a single class (accuracy ≈ 50%, EO ≈ 1.0). We fix the seed, use `Adam`, and use a modest `alpha` to keep it well-behaved, but **results still vary run to run**. On this dataset the adversarial run frequently lands **no better — or even worse** than the baseline (see the 3.10 comparison below): a faithful illustration of why adversarial fairness, though powerful in theory, is the least reliable of these methods on tabular data in practice. This is also the heaviest cell in the notebook (TensorFlow backend, dozens of epochs).

In [ ]:
tf.keras.utils.set_random_seed(42)   # adversarial training is stochastic — pin the seed

adv = AdversarialFairnessClassifier(
    backend='tensorflow',
    predictor_model=[64, 'relu'],        # predictor network hidden layers
    adversary_model=[8, 'relu'],         # adversary network hidden layers
    predictor_optimizer='Adam',
    adversary_optimizer='Adam',
    alpha=0.2,                           # weight on the adversary (lower = gentler)
    batch_size=2**8,
    epochs=50,
    random_state=42,
)
# TF needs float32 arrays; sensitive features passed as a plain array
adv.fit(X_train_sc.values.astype('float32'), y_train.values, sensitive_features=sfb_train)
y_pred_adv = adv.predict(X_test_sc.values.astype('float32'))

print('=== AdversarialFairnessClassifier (in-processing, deep) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_adv):.3f}  (baseline {accuracy_score(y_test, y_pred):.3f})')
print(f'EO diff  : {equalized_odds_difference(y_test, y_pred_adv, sensitive_features=sfb_test):.3f}  (baseline {eo_base:.3f})')
print(f'DP diff  : {demographic_parity_difference(y_test, y_pred_adv, sensitive_features=sfb_test):.3f}  (baseline {dp_base:.3f})')

### 3.9 Mitigation — Post-processing: `ThresholdOptimizer`

Ideal when the training pipeline cannot be modified or retraining is too expensive.

#### How `ThresholdOptimizer` works

1. Takes an already-fitted classifier with `predict_proba` (our baseline Random Forest).
2. Searches for the optimal **per-group decision threshold** that minimizes a performance objective (here, `balanced_accuracy_score`)…
3. …subject to satisfying a fairness constraint (here, `equalized_odds` on the binary age attribute).

At inference time it applies the **group-specific threshold** rather than the global 0.5 default — e.g., 0.45 for one group and 0.55 for another — to equalize error rates.

**Advantages:** no retraining; works on any probabilistic classifier; interpretable.
**Limitations:** requires the sensitive attribute **at prediction time**; can only reshuffle an existing model's errors, not overcome a fundamentally biased model.

In [ ]:
mitigator = ThresholdOptimizer(
    estimator=model,
    constraints='equalized_odds',
    predict_method='predict_proba',
    objective='balanced_accuracy_score'
)

# Fit on the training fold: uses the already-trained model's probabilities, then searches
# for group-specific thresholds (binary race) that satisfy the equalized_odds constraint.
mitigator.fit(X_train, y_train, sensitive_features=sfb_train)

# Apply group-specific thresholds at prediction time (needs the sensitive attribute here)
y_pred_mit = mitigator.predict(X_test, sensitive_features=sfb_test)

print('=== ThresholdOptimizer (post-processing — equalized_odds on binary age) ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred_mit):.3f}  (baseline {accuracy_score(y_test, y_pred):.3f})')
print(f'EO diff  : {equalized_odds_difference(y_test, y_pred_mit, sensitive_features=sfb_test):.3f}  (baseline {eo_base:.3f})')
print(f'DP diff  : {demographic_parity_difference(y_test, y_pred_mit, sensitive_features=sfb_test):.3f}  (baseline {dp_base:.3f})\n')
print(classification_report(y_test, y_pred_mit, target_names=['Not Readmitted', 'Readmitted']))

In [ ]:
# Compare per-group metrics on the BINARY attribute: baseline vs ThresholdOptimizer.
# Equalized odds aims to bring the two groups' TPR and FPR closer together.
mf_bin_base = MetricFrame(metrics=metrics_dict, y_true=y_test, y_pred=y_pred,
                          sensitive_features=sfb_test)
mf_bin_mit  = MetricFrame(metrics=metrics_dict, y_true=y_test, y_pred=y_pred_mit,
                          sensitive_features=sfb_test)

print('=== Baseline — by binary group ===')
print(mf_bin_base.by_group.round(3).to_string())
print(f"\nTPR gap: {mf_bin_base.difference()['true_positive_rate']:.3f}   "
      f"FPR gap: {mf_bin_base.difference()['false_positive_rate']:.3f}")

print('\n=== ThresholdOptimizer — by binary group ===')
print(mf_bin_mit.by_group.round(3).to_string())
print(f"\nTPR gap: {mf_bin_mit.difference()['true_positive_rate']:.3f}   "
      f"FPR gap: {mf_bin_mit.difference()['false_positive_rate']:.3f}")

### 3.10 All Mitigators Side by Side

We now place every approach on the same axes, all measured on the **binary age** attribute:

- **Baseline** — vanilla Random Forest, no intervention
- **CorrelationRemover** — pre-processing
- **ExponentiatedGradient** / **GridSearch** / **Adversarial** — in-processing
- **ThresholdOptimizer** — post-processing

Fairness mitigation almost never comes free: equalizing error rates across groups usually
costs some aggregate accuracy. The **Pareto scatter** below makes that tradeoff concrete —
the ideal point is the **top-left** (high accuracy, low equalized-odds difference). No single
method dominates; the right choice depends on your deployment constraints (Can you retrain? Is
the sensitive attribute available at inference?) and on **which fairness criterion** your context
demands — itself a value judgment.

In [ ]:
# Collect every model's test predictions (all evaluated on the binary race attribute)
runs = {
    'Baseline':              (y_pred,     'coral',      'pre/none'),
    'CorrelationRemover':    (y_pred_cr,  'orange',     'pre-processing'),
    'ExponentiatedGradient': (y_pred_eg,  'seagreen',   'in-processing'),
    'GridSearch':            (y_pred_gs,  'mediumseagreen', 'in-processing'),
    'Adversarial':           (y_pred_adv, 'darkturquoise',  'in-processing'),
    'ThresholdOptimizer':    (y_pred_mit, 'violet',     'post-processing'),
}

rows = []
for name, (yp, _c, family) in runs.items():
    rows.append({
        'model':    name,
        'family':   family,
        'accuracy': accuracy_score(y_test, yp),
        'dp_diff':  demographic_parity_difference(y_test, yp, sensitive_features=sfb_test),
        'eo_diff':  equalized_odds_difference(y_test, yp, sensitive_features=sfb_test),
    })
comparison = pd.DataFrame(rows).set_index('model').round(4)

print('=== ALL METHODS — binary age attribute ===')
print(comparison.to_string())
print('\nLower dp_diff / eo_diff = fairer  |  higher accuracy = better')

# --- Grouped bar chart: accuracy, DP diff, EO diff ---
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Mitigation Comparison — Accuracy & Fairness (binary age)', fontsize=14)

panel = [
    ('accuracy', 'Accuracy\n(higher = better)',          None),
    ('dp_diff',  'DP Difference\n(lower = fairer)',       0.10),
    ('eo_diff',  'EO Difference\n(lower = fairer)',       0.10),
]
bar_colors = [runs[m][1] for m in comparison.index]
for ax, (metric, title, thresh) in zip(axes, panel):
    vals = comparison[metric]
    bars = ax.bar(range(len(vals)), vals.values, color=bar_colors)
    if thresh is not None:
        ax.axhline(thresh, color='yellow', linestyle='--', linewidth=1.3, label=f'Threshold ({thresh})')
        ax.legend(fontsize=8)
    ax.set_title(title)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=35, ha='right', fontsize=8)
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

# --- Pareto scatter: accuracy vs EO difference (ideal = top-left) ---
fig, ax = plt.subplots(figsize=(9, 6))
for name, (yp, color, _f) in runs.items():
    ax.scatter(comparison.loc[name, 'eo_diff'], comparison.loc[name, 'accuracy'],
               s=220, color=color, edgecolor='white', zorder=4, label=name)
    ax.annotate(name, (comparison.loc[name, 'eo_diff'], comparison.loc[name, 'accuracy']),
                textcoords='offset points', xytext=(8, 6), fontsize=9)
ax.set_xlabel('Equalized-odds difference  (← fairer)')
ax.set_ylabel('Accuracy  (higher = better ↑)')
ax.set_title('Accuracy–Fairness Tradeoff — ideal is TOP-LEFT')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

## 4. Review

### Part 2 — SHAP plots used

| Plot | API | What it shows |
|---|---|---|
| Bar | `shap.plots.bar` | Mean \|SHAP\| — quick global ranking |
| Dot / Violin | `shap.summary_plot` | All samples: importance, direction, nonlinearity |
| Waterfall | `shap.plots.waterfall` | Step-by-step explanation of one prediction |
| Force | `shap.force_plot` | Compact one-prediction decomposition; TN/TP/FP/FN comparison |
| Dependence | `shap.dependence_plot` | Feature effect + auto interaction coloring |
| Decision | `shap.decision_plot` | Prediction paths, misclassified highlighted |
| Heatmap | `shap.plots.heatmap` | Dataset-level SHAP matrix, sorted by output |

**SHAP core identities:**
- $f(x) = \phi_0 + \sum_j \phi_j$ — predictions decompose exactly
- $\phi_j > 0$ → feature $j$ pushed this prediction above the base rate; $\phi_j < 0$ → below it
- `shap_values(X)` returns `(n, p, 2)` in SHAP ≥ 0.44 — index `[:, :, 1]` for class 1

### Part 3 — Fairlearn tools used

| Tool | Module | Family | Mechanism |
|---|---|---|---|
| `MetricFrame` | `fairlearn.metrics` | Assessment | Disaggregated audit — full metric table per group, with `.difference()` / `.ratio()` |
| `demographic_parity_*` / `equalized_odds_*` | `fairlearn.metrics` | Assessment | Scalar summaries of the same gaps |
| `CorrelationRemover` | `fairlearn.preprocessing` | Pre-processing | Strip linear correlation between features and the sensitive attribute |
| `ExponentiatedGradient` | `fairlearn.reductions` | In-processing | Constrained-optimization game → one fair randomized model |
| `GridSearch` | `fairlearn.reductions` | In-processing | Sweep penalty weights → Pareto frontier of tradeoffs |
| `AdversarialFairnessClassifier` | `fairlearn.adversarial` | In-processing | Predictor vs. adversary networks (needs TF/PyTorch) |
| `ThresholdOptimizer` | `fairlearn.postprocessing` | Post-processing | Group-specific decision thresholds on a fixed model |

**Choosing a mitigation family:**

| If you… | Use | Note |
|---|---|---|
| control the data pipeline, want model-agnostic fairness | **Pre-processing** (`CorrelationRemover`) | Removes only *linear* proxy signal |
| can retrain and want the strongest accuracy–fairness balance | **In-processing** (reductions / adversarial) | Sensitive attribute needed at train time only |
| cannot retrain (legacy / vendor model) | **Post-processing** (`ThresholdOptimizer`) | Needs the sensitive attribute **at inference** |

### Takeaways

- **SHAP and Fairlearn answer different questions about the same model.** SHAP explains *why*
  one prediction came out the way it did; Fairlearn asks whether the *pattern* of predictions
  is equitable across groups — a model can be perfectly explainable and still unfair, or hard to
  explain and still fair.
- **Dropping the protected attribute is not enough** — proxies leak through correlated clinical
  variables. That's why mitigation, not just blindness, is required, and why the SHAP
  dependence/interaction plots above are a useful first check for suspicious proxy features.
- **Fairness mitigation almost never comes free.** Equalizing error rates across groups usually
  costs some aggregate accuracy — the Pareto scatter in 3.10 makes that tradeoff concrete. No
  mitigation family dominates; the right choice depends on deployment constraints (can you
  retrain? is the sensitive attribute available at inference?) and on which fairness criterion
  your context demands — itself a value judgment, not a technical one.
- The regression form of both tools — SHAP values in dollars, fairness audited across
  neighborhoods rather than demographics — is in `U1_RealEstate-6_Interp`.